# Parameter Golf Autoresearch Analysis

Notebook nay dung de xem nhanh tien do autoresearch tu `results.tsv` va cac run folders trong `autoresearch/runs/`.

No tap trung vao:
- baseline, best run, keep/discard/crash rate
- `val_bpb` theo thu tu run
- artifact size theo run
- thong tin them tu `metrics.json` neu co


In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd


def locate_results_tsv() -> Path:
    candidates = [
        Path("results.tsv"),
        Path("autoresearch/results.tsv"),
        Path.cwd() / "results.tsv",
        Path.cwd() / "autoresearch" / "results.tsv",
    ]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError("Could not find results.tsv")


def load_results() -> tuple[pd.DataFrame, Path]:
    results_path = locate_results_tsv()
    df = pd.read_csv(results_path, sep="\t")
    if "status" in df.columns:
        df["status"] = df["status"].fillna("").astype(str).str.strip().str.upper()
    for col in ["val_bpb", "artifact_bytes"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.reset_index(drop=True)
    df["run_idx"] = range(len(df))
    return df, results_path


def load_run_metrics(results_path: Path) -> pd.DataFrame:
    runs_dir = results_path.parent / "runs"
    rows = []
    if not runs_dir.exists():
        return pd.DataFrame()
    for metrics_file in sorted(runs_dir.glob("*/metrics.json")):
        if metrics_file.parent.name == "TEMPLATE_RUN":
            continue
        try:
            payload = json.loads(metrics_file.read_text(encoding="utf-8"))
        except Exception:
            continue
        payload["run_id"] = metrics_file.parent.name
        rows.append(payload)
    return pd.DataFrame(rows)


df, results_path = load_results()
metrics_df = load_run_metrics(results_path)

print(f"results.tsv: {results_path}")
print(f"Total runs: {len(df)}")
print(f"Columns: {list(df.columns)}")
display(df.head(10))
if not metrics_df.empty:
    print(f"Run metrics found: {len(metrics_df)}")
    display(metrics_df.head(10))


In [ ]:
if df.empty:
    print("results.tsv is empty except for the header.")
else:
    counts = df["status"].value_counts(dropna=False)
    print("Status counts:")
    print(counts.to_string())

    n_keep = counts.get("KEEP", 0) + counts.get("BASELINE", 0) + counts.get("CANDIDATE", 0)
    n_discard = counts.get("DISCARD", 0)
    n_crash = counts.get("CRASH", 0)
    n_decided = n_keep + n_discard
    if n_decided > 0:
        print(f"\nKeep-like rate: {n_keep}/{n_decided} = {n_keep / n_decided:.1%}")

    valid = df[df["val_bpb"].notna()].copy()
    if not valid.empty:
        baseline_row = valid.iloc[0]
        best_row = valid.loc[valid["val_bpb"].idxmin()]
        print(f"\nBaseline run: {baseline_row['run_id']}  val_bpb={baseline_row['val_bpb']:.6f}")
        print(f"Best run:     {best_row['run_id']}  val_bpb={best_row['val_bpb']:.6f}")
        print(f"Description:  {best_row['description']}")
        print(f"Delta:        {baseline_row['val_bpb'] - best_row['val_bpb']:+.6f}")


In [ ]:
if df.empty:
    print("No runs to inspect yet.")
else:
    cols = [c for c in ["run_id", "status", "val_bpb", "artifact_bytes", "description"] if c in df.columns]
    print("Recent runs:")
    display(df[cols].tail(10))

    top = df[df["val_bpb"].notna()].sort_values("val_bpb", ascending=True)
    print("Top runs by val_bpb:")
    display(top[cols].head(10))


In [ ]:
if df.empty or df["val_bpb"].dropna().empty:
    print("Need at least one completed run with val_bpb to plot progress.")
else:
    plot_df = df[df["val_bpb"].notna()].copy()
    plot_df["running_best"] = plot_df["val_bpb"].cummin()

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    color_map = {
        "BASELINE": "#1f77b4",
        "KEEP": "#2ca02c",
        "CANDIDATE": "#17becf",
        "DISCARD": "#7f7f7f",
        "CRASH": "#d62728",
    }
    colors = [color_map.get(s, "#9467bd") for s in plot_df["status"]]

    axes[0].scatter(plot_df["run_idx"], plot_df["val_bpb"], c=colors, s=50)
    axes[0].plot(plot_df["run_idx"], plot_df["running_best"], color="black", linewidth=1.5, label="running_best")
    axes[0].set_title("val_bpb over runs")
    axes[0].set_xlabel("Run index")
    axes[0].set_ylabel("val_bpb (lower is better)")
    axes[0].grid(alpha=0.2)
    axes[0].legend()

    artifact_df = plot_df[plot_df["artifact_bytes"].notna()].copy()
    if artifact_df.empty:
        axes[1].text(0.5, 0.5, "No artifact_bytes yet", ha="center", va="center")
        axes[1].set_axis_off()
    else:
        axes[1].bar(artifact_df["run_idx"], artifact_df["artifact_bytes"] / 1_000_000, color=colors[: len(artifact_df)])
        axes[1].axhline(16.0, color="red", linestyle="--", linewidth=1, label="16MB cap")
        axes[1].set_title("Artifact size over runs")
        axes[1].set_xlabel("Run index")
        axes[1].set_ylabel("artifact size (MB)")
        axes[1].grid(alpha=0.2)
        axes[1].legend()

    plt.tight_layout()
    plt.show()


In [ ]:
if metrics_df.empty:
    print("No per-run metrics.json found yet.")
else:
    merged = df.merge(metrics_df, on="run_id", how="left", suffixes=("", "_metrics"))
    cols = [
        c
        for c in [
            "run_id",
            "status",
            "val_bpb",
            "pre_quant_val_bpb",
            "artifact_bytes",
            "train_time_ms",
            "eval_time_ms",
            "step_stop",
            "description",
        ]
        if c in merged.columns
    ]
    display(merged[cols].sort_values("run_id"))
